# AI Jira Story Generator using LangGraph & Streamlit

This notebook contains the complete code to run the Jira Story Generator agent right here in Google Colab.
Since Colab doesn't natively expose ports for web servers, we will use `localtunnel` to expose the Streamlit web application so you can interact with it.

## 1. Install Dependencies
First, we need to install the required Python libraries for the AI Agent and `localtunnel` (via npm) for exposing the Streamlit app.

In [ ]:
!pip install -q streamlit langchain langchain-openai langgraph
!npm install -g localtunnel

## 2. Create the Application File
We use the `%%writefile` magic command to save our LangGraph agent and Streamlit UI code into a single file named `app.py` directly on the Colab filesystem. 

### 🧠 How it works:
1. **AgentState**: A dictionary that tracks the current state of our graph (holding user input and the resulting story).
2. **generate_story**: A Python function (node) that uses `ChatOpenAI` and `ChatPromptTemplate` to take the user input and transform it into a well-formatted Jira story.
3. **StateGraph**: We wire our `generate_story` node into a graph, compile it, and expose it as `agent_app`.
4. **Streamlit UI**: We build the frontend that accepts the OpenAI API key, text input, and invokes the `agent_app`.

In [ ]:
%%writefile app.py
import os
import streamlit as st
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# ==========================================
# 1. LANGGRAPH AGENT DEFINITION
# ==========================================

# Define the state of our agent to hold input and output
class AgentState(TypedDict):
    user_input: str
    jira_story: str

# Node function to generate the Jira story
def generate_story(state: AgentState):
    # Initialize the LLM (Requires OPENAI_API_KEY environment variable)
    llm = ChatOpenAI(temperature=0.7, model="gpt-4o") 
    
    # Define the prompt template for the AI
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert Agile Product Owner. Your task is to write a comprehensive Jira user story based on the user's input. 
        Please format your response using Markdown.
        
        The story MUST include:
        1. **Title:** A clear and concise title for the story.
        2. **User Story:** As a [role], I want [feature] so that [benefit].
        3. **Description:** A detailed explanation of the feature or requirement.
        4. **Acceptance Criteria:** A numbered list of conditions that must be met for the story to be considered complete. Use BDD format (Given, When, Then) if applicable.
        """),
        ("user", "{user_input}")
    ])
    
    # Create the chain
    chain = prompt | llm
    
    # Invoke the LLM
    response = chain.invoke({"user_input": state["user_input"]})
    
    return {"jira_story": response.content}

# Build the LangGraph
workflow = StateGraph(AgentState)
workflow.add_node("generate_story", generate_story)
workflow.set_entry_point("generate_story")
workflow.add_edge("generate_story", END)
agent_app = workflow.compile()

# ==========================================
# 2. STREAMLIT APP DEFINITION
# ==========================================

st.set_page_config(page_title="Jira Story Generator", page_icon="📝", layout="centered")
st.title("📝 AI Jira Story Generator")
st.markdown("This app uses a **LangGraph agent** to automatically write comprehensive Jira user stories.")

with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input("OpenAI API Key", type="password", help="Enter your OpenAI API Key.")
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key

st.subheader("What do you need a story for?")
user_input = st.text_area(
    "Describe your feature or requirement in plain text:",
    height=150,
    placeholder="Example: I need a login page for our app..."
)

if st.button("Generate Jira Story", type="primary"):
    if not api_key:
        st.error("⚠️ Please enter your OpenAI API Key in the sidebar.")
    elif not user_input.strip():
        st.warning("⚠️ Please provide a description for the story.")
    else:
        with st.spinner("🧠 Agent is writing your Jira story..."):
            try:
                result = agent_app.invoke({"user_input": user_input, "jira_story": ""})
                st.success("✨ Story Generated Successfully!")
                st.markdown("---")
                with st.container(border=True):
                    st.markdown(result["jira_story"])
            except Exception as e:
                st.error(f"❌ An error occurred: {str(e)}")


## 3. Get the Localtunnel Password
Localtunnel requires a password to verify you are a real user. Run this cell to get the IP address of this Colab instance. 

👉 **Copy the IP address that prints below** as you will need to paste it into the localtunnel warning page in the final step.

In [ ]:
import urllib.request
print("Your Endpoint IP / Password for localtunnel is:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

## 4. Run the App
Run this cell to start Streamlit and `localtunnel` simultaneously. 

**Follow these instructions:**
1. Wait a few seconds for an output that says `your url is: https://some-random-words.loca.lt`.
2. Click that link to open it in a new tab.
3. On the webpage that opens, paste the **IP address** from Step 3 into the "Endpoint IP" box.
4. Click "Click to Continue" to start using your Streamlit app!

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501